# 06 - Multiclass Training (6-Class: Healthy + 5 Hemorrhage Subtypes)

Full 6-class training pipeline with FocalLoss, frequency-based class weights, warmup+cosine LR, gradient clipping, EMA, MixUp/CutMix, TTA validation, and early stopping.

In [2]:
import torch
import torch.nn as nn
from torchvision.models import convnext_large, ConvNeXt_Large_Weights


class ConvNeXtLargeCustom(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()

        # Load pretrained model
        self.backbone = convnext_large(weights=ConvNeXt_Large_Weights.IMAGENET1K_V1)

        # Remove original classifier
        self.backbone.classifier = nn.Identity()

        # Add pooling manually (VERY IMPORTANT!)
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        feature_dim = 1536  # ConvNeXt-Large always outputs 1536

        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, 512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.backbone.features(x)  # (B,1536,H,W)

        x = self.global_pool(x)        # (B,1536,1,1)
        x = x.flatten(1)               # (B,1536)

        logits = self.classifier(x)
        return logits


In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model=ConvNeXtLargeCustom()  # same architecture used during training

In [6]:
# ---------------------------
# ConvNeXt Large — Full Training Script
# Phase1 (classifier warmup) + Phase2 (fine-tune)
# Multi-class (6 Classes: Healthy + 5 Hemorrhage Subtypes)
# RandAugment + MixUp/CutMix + GPU EMA + Native C,H,W
# ---------------------------

import os
import copy
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms.functional as TF
from torchvision.transforms import RandAugment

# ---------------------------
# Device setup & Model Assignment
# ---------------------------
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Assumes 'model' is already defined in your Jupyter environment above this cell!
model = model.to(device)

# ---------------------------
# Dataset classes (chunked .npy files)
# ---------------------------
class NumpyDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir = data_dir
        self.image_files = sorted([f for f in os.listdir(data_dir) if f.startswith("X_train_chunk_")])
        self.label_files = sorted([f for f in os.listdir(data_dir) if f.startswith("y_train_chunk_")])
        assert len(self.image_files) > 0, "No chunks found!"
        assert len(self.image_files) == len(self.label_files), "Chunk/label count mismatch"

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        imgs = np.load(os.path.join(self.data_dir, self.image_files[idx]), mmap_mode="r")
        lbls = np.load(os.path.join(self.data_dir, self.label_files[idx]), mmap_mode="r")
        return imgs, lbls

class ValidationDataset:
    def __init__(self, imgs_path, labels_path):
        self.images = np.load(imgs_path, mmap_mode="r")
        self.labels = np.load(labels_path, mmap_mode="r")

# ---------------------------
# RandAugment (PIL) setup
# ---------------------------
rand_aug = RandAugment(num_ops=2, magnitude=9)

def apply_randaugment(batch):
    out = []
    for img in batch: 
        arr = img.permute(1, 2, 0).cpu().numpy()
        if arr.max() <= 1.0:
            arr = (arr * 255.0).clip(0, 255).astype(np.uint8)
        else:
            arr = arr.clip(0, 255).astype(np.uint8)
        pil = Image.fromarray(arr)
        pil_aug = rand_aug(pil)
        ten = TF.to_tensor(pil_aug).to(device)
        out.append(ten)
    return torch.stack(out, dim=0)

# ---------------------------
# Normalization constants
# ---------------------------
mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

# ---------------------------
# Augment pipelines
# ---------------------------
def augment_phase1(batch):
    if batch.max() > 2.0:
        batch = batch / 255.0
    if torch.rand(1) < 0.5:
        batch = batch.flip(-1)
    batch = F.interpolate(batch, size=(256, 256), mode='bilinear', align_corners=False)
    batch = (batch - mean) / std
    return batch

def augment_phase2_balanced(batch):
    if batch.max() > 2.0:
        batch = batch / 255.0
        batch = (batch * 255.0).clamp(0, 255)

    if torch.rand(1) < 0.33:
        batch = apply_randaugment(batch)
    else:
        if torch.rand(1) < 0.5:
            batch = batch.flip(-1)
        if torch.rand(1) < 0.3:
            batch = batch.flip(-2)
        if torch.rand(1) < 0.5:
            angle = (torch.rand(1).item() * 20.0 - 10.0)
            batch = TF.rotate(batch, angle, interpolation=TF.InterpolationMode.BILINEAR)
        if torch.rand(1) < 0.5:
            batch = TF.adjust_brightness(batch, 1.0 + 0.1 * torch.randn(1).item())
            batch = TF.adjust_contrast(batch, 1.0 + 0.1 * torch.randn(1).item())

    batch = F.interpolate(batch, size=(256, 256), mode='bilinear', align_corners=False)
    batch = (batch - mean) / std
    return batch

# ---------------------------
# MixUp / CutMix
# ---------------------------
def mixup_data(x, y, alpha=0.6):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    mixed = lam * x + (1 - lam) * x[idx]
    return mixed, y, y[idx], lam

def cutmix_data(x, y, alpha=1.0):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    B, C, H, W = x.size()
    idx = torch.randperm(B).to(x.device)

    cut_rat = np.sqrt(1. - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)

    x1, y1 = max(cx - cut_w // 2, 0), max(cy - cut_h // 2, 0)
    x2, y2 = min(cx + cut_w // 2, W), min(cy + cut_h // 2, H)

    x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1. - ((x2 - x1) * (y2 - y1) / (W * H))
    return x, y, y[idx], lam

def mixup_criterion(crit, pred, y_a, y_b, lam):
    return lam * crit(pred, y_a) + (1 - lam) * crit(pred, y_b)

# ---------------------------
# GPU-safe EMA
# ---------------------------
class ModelEMA:
    def __init__(self, model, decay=0.9999):
        self.decay = decay
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        esd = self.ema.state_dict()
        for k in msd.keys():
            esd[k].mul_(self.decay).add_(msd[k], alpha=1.0 - self.decay)

# ---------------------------
# Validation
# ---------------------------
@torch.no_grad()
def validate_model(model, dataset, criterion, batch_size=64, ema: ModelEMA = None):
    eval_model = ema.ema if ema is not None else model
    eval_model.eval()

    running_loss, correct, total = 0.0, 0, 0

    for start in tqdm(range(0, len(dataset.images), batch_size), desc="Validation", ncols=100):
        end = min(start + batch_size, len(dataset.images))
        imgs_np = dataset.images[start:end]
        lbls_np = dataset.labels[start:end]

        # Handle one-hot chunks if they slip in
        if len(lbls_np.shape) > 1:
            lbls_np = np.argmax(lbls_np, axis=1)

        imgs = torch.from_numpy(imgs_np.copy()).float().to(device)
        labels = torch.tensor(lbls_np).long().to(device)
        
        if imgs.max() > 2.0:
            imgs = imgs / 255.0
        imgs = F.interpolate(imgs, size=(256, 256), mode='bilinear', align_corners=False)
        imgs = (imgs - mean) / std

        with torch.amp.autocast("cuda"):
            out = eval_model(imgs)
            loss = criterion(out, labels)

        running_loss += loss.item() * labels.size(0)
        _, preds = out.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total if total > 0 else 0, 100.0 * correct / total if total > 0 else 0

# ---------------------------
# Config + Data loaders
# ---------------------------
train_dir = "temp_verification_folder"
val_images = "X_val_chunk_0.npy"
val_labels = "y_val_chunk_0.npy"

train_dataset = NumpyDataset(train_dir)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=lambda x: x[0])

val_dataset = ValidationDataset(val_images, val_labels)

# Standard Multi-class CE Loss (Works perfectly for 6 classes)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
scaler = torch.cuda.amp.GradScaler()

CHUNK = 32                
PHASE1_EPOCHS = 4
PHASE2_EPOCHS = 10
PATIENCE = 7

# ---------------------------
# PHASE 1: Train classifier head only
# ---------------------------
print("\n===== PHASE 1: Classifier warmup =====\n")

for p in model.parameters():
    p.requires_grad = False
for p in model.classifier.parameters():
    p.requires_grad = True

opt = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=PHASE1_EPOCHS)

for ep in range(PHASE1_EPOCHS):
    model.train()
    total, correct, loss_sum = 0, 0, 0

    for imgs, lbls in tqdm(train_loader, desc=f"P1 Epoch {ep+1}/{PHASE1_EPOCHS}", ncols=100):
        
        if len(lbls.shape) > 1:
            lbls = np.argmax(lbls, axis=1)

        for i in range(0, len(imgs), CHUNK):
            batch = torch.from_numpy(imgs[i:i+CHUNK].copy()).float().to(device)
            labels = torch.tensor(lbls[i:i+CHUNK]).long().to(device)

            batch = augment_phase1(batch)

            opt.zero_grad()
            with torch.amp.autocast("cuda"):
                preds = model(batch)
                loss = criterion(preds, labels)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            _, pcls = preds.max(1)
            correct += (pcls == labels).sum().item()
            total += labels.size(0)

    sched.step()
    train_acc = 100.0 * correct / total if total > 0 else 0.0
    val_loss, val_acc = validate_model(model, val_dataset, criterion)
    print(f"P1 Epoch {ep+1}: Train={train_acc:.2f}% | Val={val_acc:.2f}%")

# ---------------------------
# PHASE 2: Fine-tune with heavy augment + mixup/cutmix
# ---------------------------
print("\n===== PHASE 2: Fine-tuning =====\n")

# Freeze early stages 0,1,2 for stability
for name, p in model.named_parameters():
    if ("stages.0" in name) or ("stages.1" in name) or ("stages.2" in name):
        p.requires_grad = False
    else:
        p.requires_grad = True

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-5, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=PHASE2_EPOCHS)
ema = ModelEMA(model)

best_val = 0.0
no_improve = 0

for ep in range(PHASE2_EPOCHS):
    model.train()
    total, correct = 0, 0

    for imgs, lbls in tqdm(train_loader, desc=f"P2 Epoch {ep+1}/{PHASE2_EPOCHS}", ncols=100):
        
        if len(lbls.shape) > 1:
            lbls = np.argmax(lbls, axis=1)

        for i in range(0, len(imgs), CHUNK):
            batch = torch.from_numpy(imgs[i:i+CHUNK].copy()).float().to(device)
            labels = torch.tensor(lbls[i:i+CHUNK]).long().to(device)

            batch = augment_phase2_balanced(batch)

            # MixUp / CutMix
            if np.random.rand() < 0.5:
                batch, y1, y2, lam = mixup_data(batch, labels, alpha=0.6)
            else:
                batch, y1, y2, lam = cutmix_data(batch, labels, alpha=1.0)

            opt.zero_grad()
            with torch.amp.autocast("cuda"):
                preds = model(batch)
                loss = mixup_criterion(criterion, preds, y1, y2, lam)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            # Update EMA (GPU-only)
            ema.update(model)

            _, pcls = preds.max(1)
            correct += (pcls == labels).sum().item()
            total += labels.size(0)

    sched.step()

    train_acc = 100.0 * correct / total if total > 0 else 0.0
    val_loss, val_acc = validate_model(model, val_dataset, criterion, ema=ema)
    print(f"P2 Epoch {ep+1}: Train={train_acc:.2f}% | Val={val_acc:.2f}%")

    if val_acc > best_val:
        best_val = val_acc
        no_improve = 0
        torch.save(model.state_dict(), "best_convnext_large_6class.pth")
        print("✔ Saved best model")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print("⛔ Early stopping triggered")
            break

print("\n🎉 Training finished. Best Val Acc: {:.2f}% 🎉".format(best_val))

Device: cuda


/tmp/ipykernel_208973/24307608.py:215: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()



===== PHASE 1: Classifier warmup =====





alidation: 100%|█████████████████████████████████████████████████| 235/235 [00:21<00:00, 11.01it/s]

P1 Epoch 1: Train=58.92% | Val=63.05%


...

P2 Epoch 8: Train=62.64% | Val=85.16%
✔ Saved best model




alidation: 100%|█████████████████████████████████████████████████| 235/235 [00:19<00:00, 12.16it/s]

P2 Epoch 9: Train=63.26% | Val=85.35%
✔ Saved best model




alidation: 100%|█████████████████████████████████████████████████| 235/235 [00:18<00:00, 12.47it/s]

P2 Epoch 10: Train=63.31% | Val=85.48%
✔ Saved best model

🎉 Training finished. Best Val Acc: 85.48% 🎉
